# 11 深度學習 — 練習

用松柏護理之家退伍軍人症資料練習 PyTorch 二元分類。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["severe_outcome"] = ((df["hospitalized"] == 1) | (df["outcome"] == "dead")).astype(int)

num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = [
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]

X_df = pd.get_dummies(df[num_cols + cat_cols + bin_cols], drop_first=True)
X_np = X_df.values.astype(np.float32)
scaler = StandardScaler()
X_np[:, 0] = scaler.fit_transform(X_np[:, 0:1]).ravel()

idx = np.arange(len(X_np))
np.random.shuffle(idx)
split = int(0.7 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]

## 題目 1：改變架構

1. 把隱藏層從 `32 → 16` 改成 `64 → 32 → 16`（三層隱藏層）
2. 計算新模型的參數量
3. 用相同的早停法訓練，比較 Val AUC
4. 更複雜的架構是否表現更好？參數/樣本比如何？

In [ ]:
# TODO: 建立 3 層隱藏層模型
# TODO: 計算參數量
# TODO: 訓練 + 早停
# TODO: 評估 Val AUC

## 題目 2：Task B — 預測重症

1. 把目標變數改成 `severe_outcome`
2. 用 `input → 32 → 16 → 1` 架構訓練
3. 畫出學習曲線（train/val loss）
4. 計算 Val AUC，與 Task A 比較

In [ ]:
# TODO: y = severe_outcome
# TODO: 建立模型、訓練、早停
# TODO: 學習曲線
# TODO: Val AUC

## 題目 3（挑戰題）：加入 Dropout 正則化

1. 在每個 ReLU 後加上 `nn.Dropout(0.3)`
2. 訓練 + 早停
3. 比較有 Dropout vs 無 Dropout 的：
   - Train AUC vs Val AUC gap
   - 學習曲線形狀
4. Dropout 是否有效減少過擬合？

In [ ]:
# TODO: 建立含 Dropout 的模型
# TODO: 訓練 + 早停
# TODO: 比較 Train-Val AUC gap
# TODO: 解讀

## 題目 4：COVID-19 表格資料小型 MLP（COVID-19 情境）

用 PyTorch 建立小型神經網路預測 COVID-19 重症。

1. 標準化特徵、切分訓練/驗證集
2. 建立小型 MLP（`nn.Sequential`），用 `BCEWithLogitsLoss` + `Adam` 訓練 ~150 epoch
3. 計算 train / validation ROC-AUC
4. 解讀：表格小資料上，DL 相對邏輯斯迴歸有明顯優勢嗎？

In [ ]:
# COVID-19：表格資料的小型 MLP 二元分類（重症）
rng = np.random.default_rng(1104)
n = 900
age = rng.integers(20, 90, n); male = rng.integers(0, 2, n)
diabetes = rng.integers(0, 2, n); vaccinated = rng.binomial(1, 0.6, n)
logit = -6 + 0.06*age + 0.4*male + 0.7*diabetes - 1.2*vaccinated
severe = rng.binomial(1, 1/(1+np.exp(-logit)))
X = np.c_[age, male, diabetes, vaccinated].astype(float); y = severe.astype(float)
print(f"COVID: n={n}, 重症比例={y.mean():.1%}, 特徵數={X.shape[1]}")

# TODO: 用 StandardScaler 標準化 X，train_test_split（stratify=y, test_size=0.3）
# TODO: 用 torch 建立小型 MLP：nn.Sequential(Linear->ReLU->Linear->ReLU->Linear(…,1))
# TODO: 用 BCEWithLogitsLoss + Adam 訓練 ~150 個 epoch（full-batch 即可）
# TODO: 用 sigmoid 取得機率，算 train / validation ROC-AUC
# TODO: 解讀：這種表格小資料，DL 相對邏輯斯迴歸有明顯優勢嗎？

## 題目 5：登革熱重症 MLP（登革熱情境）

用小型 MLP 預測重症登革熱 DHF。

1. 標準化、切分後訓練一個 1 層隱藏層的小型 MLP
2. 計算 validation ROC-AUC，並與第 10 章隨機森林比較

In [ ]:
# 登革熱：重症 DHF 的小型 MLP
rng = np.random.default_rng(1105)
n = 800
age = rng.integers(1, 80, n); secondary = rng.binomial(1, 0.45, n)
platelet = rng.normal(180, 60, n).clip(20, 400); days = rng.integers(1, 8, n)
logit = -2.5 + 1.6*secondary - 0.012*platelet + 0.15*days
dhf = rng.binomial(1, 1/(1+np.exp(-logit)))
X = np.c_[age, secondary, platelet, days].astype(float); y = dhf.astype(float)
print(f"Dengue: n={n}, DHF 比例={y.mean():.1%}")

# TODO: 標準化、切分後建立小型 MLP（1 層隱藏層 16 個神經元即可）訓練
# TODO: 算 validation ROC-AUC，並和你在第 10 章的隨機森林結果比較

## 題目 6：流感住院小型 NN（流感情境）

用小型神經網路預測流感住院。

1. 標準化、切分、訓練小型 MLP
2. 回報 validation ROC-AUC

In [ ]:
# 流感：住院預測小型 NN
rng = np.random.default_rng(1106)
n = 850
age = rng.integers(0, 95, n); chronic = rng.binomial(1, 0.25, n)
vacc = rng.binomial(1, 0.5, n); onset = rng.integers(0, 6, n)
logit = -3.5 + 0.05*age + 1.0*chronic - 0.8*vacc + 0.25*onset
hosp = rng.binomial(1, 1/(1+np.exp(-logit)))
X = np.c_[age, chronic, vacc, onset].astype(float); y = hosp.astype(float)
print(f"Flu: n={n}, 住院比例={y.mean():.1%}")

# TODO: 標準化、切分後訓練小型 MLP，回報 validation ROC-AUC

## 題目 7：結核病預後小型 NN（結核情境）

用小型神經網路預測結核病治療成功。

1. 標準化、切分、訓練小型 MLP
2. 回報 validation ROC-AUC，注意 adherence 的方向

In [ ]:
# 結核病：治療結果預後小型 NN
rng = np.random.default_rng(1107)
n = 800
age = rng.integers(18, 85, n); mdr = rng.binomial(1, 0.15, n)
hiv = rng.binomial(1, 0.1, n); adher = rng.uniform(0.4, 1.0, n)
logit = 2.0 - 1.8*mdr - 1.2*hiv + 3.0*(adher-0.7)
success = rng.binomial(1, 1/(1+np.exp(-logit)))
X = np.c_[age, mdr, hiv, adher].astype(float); y = success.astype(float)
print(f"TB: n={n}, 治療成功率={y.mean():.1%}")

# TODO: 標準化、切分後訓練小型 MLP，回報 validation ROC-AUC
# TODO: 特別注意 adherence 對成功率的方向

## 題目 8（挑戰題）：小樣本過擬合示範（模型診斷情境）

刻意用「極小樣本 + 大量雜訊特徵 + 過大網路」示範過擬合。

1. 標準化、切分（test_size=0.4）
2. 用一個過大的網路（2 層各 128 神經元）訓練較多 epoch
3. 記錄並繪製 train loss 與 validation loss 兩條曲線
4. 觀察 train loss 下降但 val loss 反轉上升，找出應早停的 epoch
5. 解讀：為何小樣本 + 大網路易過擬合？流病小資料該怎麼做？

In [ ]:
# 小樣本過擬合示範：n 很小、雜訊特徵很多（挑戰題）
rng = np.random.default_rng(1108)
n = 90                      # 極小樣本
signal = rng.normal(0, 1, n)
noise = rng.normal(0, 1, (n, 30))   # 30 個純雜訊特徵
y = (1/(1+np.exp(-(1.5*signal))) > rng.uniform(0, 1, n)).astype(float)
X = np.c_[signal, noise].astype(float)   # 1 個有訊號 + 30 個雜訊
print(f"n={n}, 特徵數={X.shape[1]}（僅 1 個有訊號），正例比例={y.mean():.1%}")

# TODO: 標準化，train_test_split（test_size=0.4）
# TODO: 用一個「過大」的網路（如 2 層各 128 神經元）訓練較多 epoch
# TODO: 記錄每個 epoch 的 train loss 與 validation loss，畫成兩條曲線
# TODO: 觀察 train loss 持續下降但 val loss 反轉上升 → 過擬合
# TODO: 解讀：為什麼小樣本 + 大網路容易過擬合？在流病小資料上你會怎麼做？